# MTL ADMET Predictor

Single-notebook version of the `mtl_nn` package. Predicts five ADMET endpoints from Morgan fingerprints with a hard-parameter-sharing multi-task network.

**Endpoints**

| Code        | TDC dataset       | Group | Type            |
| ----------- | ----------------- | ----- | --------------- |
| `caco2`     | `Caco2_Wang`      | ADME  | regression      |
| `bbb`       | `BBB_Martins`     | ADME  | classification  |
| `cyp3a4`    | `CYP3A4_Veith`    | ADME  | classification  |
| `half_life` | `Half_Life_Obach` | ADME  | regression      |
| `herg`      | `hERG`            | Tox   | classification  |

Run the cells top-to-bottom.

## 1. Install dependencies (run once)


Skip this cell if your environment already has them

In [ ]:
!pip install numpy==1.26.4 scikit-learn==1.2.2 pandas==2.2.2

In [ ]:
!pip install torch tqdm

In [ ]:
!pip install rdkit-pypi

In [ ]:
!pip install torch PyTDC scikit-learn numpy pandas tqdm rdkit

## 2. Imports

In [ ]:
pip install --upgrade --force-reinstall numpy pandas

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Iterable, List, Literal, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
from sklearn.metrics import mean_squared_error, roc_auc_score
from sklearn.model_selection import train_test_split
from tdc.single_pred import ADME, Tox
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# RDKit's C++ warnings are noisy on TDC corpora.
RDLogger.DisableLog("rdApp.*")

In [2]:
import shutil, os

cache_path = os.path.expanduser("~/.tdc_data")
if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print("Cache TDC wyczyszczony")
else:
    print("Brak cache do wyczyszczenia")

Brak cache do wyczyszczenia


## 3. Configuration

All knobs live here: which tasks to model, fingerprint shape, network shape, training hyperparameters, and CUDA toggles. `TASKS` is the single source of truth — the rest of the code iterates over it.

In [3]:
import torch
import os

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)')

CUDA: True
  GPU 0: Tesla T4 (15.6 GB)
  GPU 1: Tesla T4 (15.6 GB)


In [4]:
TaskType = Literal["classification", "regression"]
DeviceMode = Literal["auto", "cuda", "cpu"]


@dataclass(frozen=True)
class TaskSpec:
    name: str             # column name in merged DataFrame
    task_type: TaskType   # drives loss + output activation
    tdc_name: Optional[str] = None         # exact TDC dataset name
    tdc_group: Optional[str] = None        # 'ADME' or 'Tox'


TASKS: Tuple[TaskSpec, ...] = (
    TaskSpec("caco2",      "regression",     "Caco2_Wang",      "ADME"),
    TaskSpec("bbb",        "classification", "BBB_Martins",     "ADME"),
    TaskSpec("cyp3a4",     "classification", "CYP3A4_Veith",    "ADME"),
    TaskSpec("half_life",  "regression",     "Half_Life_Obach", "ADME"),
    TaskSpec("herg",       "classification", "hERG",            "Tox"),
    TaskSpec("hum_mic_cl", "regression"),
    TaskSpec("ppb",        "regression"),
    TaskSpec("logd",       "regression"),
    TaskSpec("water_sol",  "regression"),
)

# --- Featurizer ---
FP_RADIUS: int = 2
FP_NBITS: int = 2048

# --- Model ---
SHARED_HIDDEN_DIMS: Tuple[int, ...] = (1024, 512, 256)
HEAD_HIDDEN_DIM: int = 64
DROPOUT: float = 0.1 # 0.2

# --- Training ---
BATCH_SIZE: int = 256
EPOCHS: int = 100
LEARNING_RATE: float = 3e-4 # 1e-3
WEIGHT_DECAY: float = 1e-4
VAL_FRACTION: float = 0.1
TEST_FRACTION: float = 0.1
RANDOM_SEED: int = 42
                                # dodane
TASK_WEIGHTS: Dict[str, float] = {
    "caco2":      1.0,
    "bbb":        1.0,
    "cyp3a4":     0.5,
    "half_life":  4.0,
    "herg":       3.0,
    "hum_mic_cl": 1.0,
    "ppb":        1.0,
    "logd":       0.5,
    "water_sol":  0.5,
}

# --- CUDA / device ---
DEVICE_MODE: DeviceMode = "auto"   # 'auto' | 'cuda' | 'cpu'
CUDA_DEVICE_INDEX: int = 0
CUDNN_BENCHMARK: bool = True
CUDNN_DETERMINISTIC: bool = False
ALLOW_TF32: bool = True
USE_AMP: bool = True
PIN_MEMORY: Optional[bool] = None  # None => auto


def resolve_device() -> torch.device:
    """Resolve DEVICE_MODE into a concrete torch.device.

    'cuda' raises if unavailable so you don't silently fall back to CPU.
    """
    if DEVICE_MODE == "cpu":
        return torch.device("cpu")
    if DEVICE_MODE == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError(
                "DEVICE_MODE='cuda' but torch.cuda.is_available() is False. "
                "Install a CUDA-enabled PyTorch build or set DEVICE_MODE='auto'."
            )
        return torch.device(f"cuda:{CUDA_DEVICE_INDEX}")
    if torch.cuda.is_available():
        return torch.device(f"cuda:{CUDA_DEVICE_INDEX}")
    return torch.device("cpu")


def configure_cuda_runtime() -> None:
    """Apply global cuDNN/TF32 flags. No-op on CPU."""
    if not torch.cuda.is_available():
        return
    torch.backends.cudnn.benchmark = CUDNN_BENCHMARK
    torch.backends.cudnn.deterministic = CUDNN_DETERMINISTIC
    torch.backends.cuda.matmul.allow_tf32 = ALLOW_TF32
    torch.backends.cudnn.allow_tf32 = ALLOW_TF32


def use_pin_memory(device: torch.device) -> bool:
    if PIN_MEMORY is not None:
        return PIN_MEMORY
    return device.type == "cuda"


configure_cuda_runtime()
DEVICE = resolve_device()
print(f"[config] using device: {DEVICE}")

[config] using device: cuda:0


## 4. Featurizer — SMILES → Morgan fingerprints

Returns `None` for unparseable SMILES; callers drop those rows. The batch helper also returns the surviving indices so labels can be re-aligned.

In [5]:
def smiles_to_morgan_fp(
    smiles: str,
    radius: int = 2,
    n_bits: int = 2048,
) -> Optional[np.ndarray]:
    """Single SMILES -> Morgan bit vector (np.ndarray) or None on failure."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=np.float32)
    Chem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


def batch_smiles_to_fps(
    smiles_list: Iterable[str],
    radius: int = 2,
    n_bits: int = 2048,
) -> Tuple[np.ndarray, List[int]]:
    """Featurize an iterable of SMILES, returning (fps, surviving_indices)."""
    fps: List[np.ndarray] = []
    valid_indices: List[int] = []
    for i, smi in enumerate(smiles_list):
        fp = smiles_to_morgan_fp(smi, radius=radius, n_bits=n_bits)
        if fp is not None:
            fps.append(fp)
            valid_indices.append(i)
    if not fps:
        return np.zeros((0, n_bits), dtype=np.float32), []
    return np.stack(fps, axis=0), valid_indices

## 5. Download and merge the TDC datasets

Outer-merge on SMILES. Most molecules will only have a label for one task; the empty cells become `NaN` and are masked out at loss time.

In [6]:
from tdc.single_pred import ADME, Tox

_GROUP_TO_CLASS = {"ADME": ADME, "Tox": Tox}

def _fetch_one(spec: TaskSpec) -> Optional[pd.DataFrame]:
    if spec.tdc_name is None:
        return None
    cls = _GROUP_TO_CLASS[spec.tdc_group]
    df = cls(name=spec.tdc_name).get_data()
    out = df[["Drug", "Y"]].rename(columns={"Drug": "smiles", "Y": spec.name})
    out = out.groupby("smiles", as_index=False)[spec.name].mean()
    if spec.task_type == "classification":
        out = out[out[spec.name].isin([0.0, 1.0])]
    
    if spec.name == "half_life":
        out[spec.name] = np.log1p(out[spec.name])
    return out


def download_and_merge_tdc(tasks: Sequence[TaskSpec]) -> pd.DataFrame:
    merged: Optional[pd.DataFrame] = None
    for spec in tasks:
        if spec.tdc_name is None:
            continue
        print(f"[tdc] downloading {spec.tdc_name} ...")
        part = _fetch_one(spec)
        merged = part if merged is None else merged.merge(part, on="smiles", how="outer")
    return merged

In [7]:
from tdc.single_pred import ADME
data = ADME(name="Half_Life_Obach")
df_diag = data.get_data()
print(df_diag["Y"].describe())
print(df_diag["Y"].quantile([0.90, 0.95, 0.99]))

Found local copy...
Loading...
Done!


count     667.000000
mean       18.212916
std        81.869185
min         0.065000
25%         1.750000
50%         4.200000
75%        11.000000
max      1200.000000
Name: Y, dtype: float64
0.90     30.4
0.95     45.7
0.99    200.0
Name: Y, dtype: float64


In [8]:
from rdkit import Chem

def canon(smi):
    mol = Chem.MolFromSmiles(str(smi))
    return Chem.MolToSmiles(mol) if mol else None

In [9]:
base_url = "https://raw.githubusercontent.com/mindrank-ai/PharmaBench/main/data/final_datasets/"

# Słownik: nazwa w naszym modelu → plik PharmaBench
pharma_files = {
    "hum_mic_cl": "hum_mic_cl_reg_final_data.csv",
    "ppb":        "ppb_reg_final_data.csv",
    "logd":       "logd_reg_final_data.csv",
    "water_sol":  "water_sol_reg_final_data.csv",
}

dfs = {}
for name, fname in pharma_files.items():
    df = pd.read_csv(f"{base_url}{fname}")
    df = df[["Smiles_unify", "value"]].rename(
        columns={"Smiles_unify": "smiles", "value": name}
    )
    df["smiles"] = df["smiles"].apply(canon)
    df = df.dropna(subset=["smiles", name])
    print(f"{name}: {len(df)} związków, wartości: min={df[name].min():.2f} max={df[name].max():.2f}")
    dfs[name] = df

hum_mic_cl: 2286 związków, wartości: min=-2.52 max=6.36
ppb: 1262 związków, wartości: min=0.00 max=1.00
logd: 13068 związków, wartości: min=-4.80 max=9.13
water_sol: 11701 związków, wartości: min=-9.22 max=10.14


In [10]:
df_merged = download_and_merge_tdc(TASKS)

for name, df in dfs.items():
    df_merged = df_merged.merge(df, on="smiles", how="outer")

print(f"Łączna liczba cząsteczek: {len(df_merged)}")
for spec in TASKS:
    n = df_merged[spec.name].notna().sum()
    print(f"  {spec.name}: {n} labeled ({n/len(df_merged):.1%})")

Found local copy...
Loading...
Done!
Found local copy...
Loading...
Done!
Found local copy...
Loading...
Done!
Found local copy...
Loading...
Done!
Found local copy...
Loading...
Done!


[tdc] downloading Caco2_Wang ...
[tdc] downloading BBB_Martins ...
[tdc] downloading CYP3A4_Veith ...
[tdc] downloading Half_Life_Obach ...
[tdc] downloading hERG ...
Łączna liczba cząsteczek: 41563
  caco2: 906 labeled (2.2%)
  bbb: 1965 labeled (4.7%)
  cyp3a4: 12328 labeled (29.7%)
  half_life: 665 labeled (1.6%)
  herg: 645 labeled (1.6%)
  hum_mic_cl: 2286 labeled (5.5%)
  ppb: 1262 labeled (3.0%)
  logd: 13068 labeled (31.4%)
  water_sol: 11701 labeled (28.2%)


## 6. PyTorch Dataset and DataLoaders

Each item yields `(fingerprint, target_vector, mask_vector)`. The mask is precomputed here so the loss function stays simple. NaNs are replaced with `0.0` *after* the mask is built — keeping NaNs in tensors corrupts gradients even when zeroed by the mask.

In [11]:
class ADMETMultiTaskDataset(Dataset):
    def __init__(
        self,
        fingerprints: np.ndarray,   # (N, n_bits) float32
        targets: np.ndarray,        # (N, n_tasks) float32, NaN for missing
        mask: np.ndarray,           # (N, n_tasks) float32, 1 where labeled
    ) -> None:
        assert fingerprints.shape[0] == targets.shape[0] == mask.shape[0]
        targets = np.nan_to_num(targets, nan=0.0).astype(np.float32)
        self.fingerprints = torch.from_numpy(fingerprints.astype(np.float32))
        self.targets = torch.from_numpy(targets)
        self.mask = torch.from_numpy(mask.astype(np.float32))

    def __len__(self) -> int:
        return self.fingerprints.shape[0]

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        return self.fingerprints[idx], self.targets[idx], self.mask[idx]


def _featurize_frame(
    df: pd.DataFrame,
    tasks: Sequence[TaskSpec],
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    smiles = df["smiles"].tolist()
    fps, valid_idx = batch_smiles_to_fps(smiles, radius=FP_RADIUS, n_bits=FP_NBITS)
    df_valid = df.iloc[valid_idx].reset_index(drop=True)
    target_cols = [t.name for t in tasks]
    targets = df_valid[target_cols].to_numpy(dtype=np.float32)
    mask = (~np.isnan(targets)).astype(np.float32)
    return fps, targets, mask


def build_dataloaders(
    df: pd.DataFrame,
    tasks: Sequence[TaskSpec],
    batch_size: int = BATCH_SIZE,
    val_fraction: float = VAL_FRACTION,
    test_fraction: float = TEST_FRACTION,
    seed: int = RANDOM_SEED,
    num_workers: int = 0,
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    fps, targets, mask = _featurize_frame(df, tasks)
    print(f"[dataset] featurized {fps.shape[0]} / {len(df)} molecules (rest unparseable)")

    idx = np.arange(fps.shape[0])
    idx_train, idx_temp = train_test_split(
        idx, test_size=val_fraction + test_fraction, random_state=seed
    )
    rel_test = test_fraction / (val_fraction + test_fraction)
    idx_val, idx_test = train_test_split(idx_temp, test_size=rel_test, random_state=seed)

    device = resolve_device()
    pin = use_pin_memory(device)

    def make(indices: np.ndarray, shuffle: bool) -> DataLoader:
        ds = ADMETMultiTaskDataset(fps[indices], targets[indices], mask[indices])
        return DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=pin,
        )

    return make(idx_train, True), make(idx_val, False), make(idx_test, False)


train_loader, val_loader, test_loader = build_dataloaders(df_merged, TASKS)
print(f"train batches: {len(train_loader)}  val: {len(val_loader)}  test: {len(test_loader)}")

[dataset] featurized 41563 / 41563 molecules (rest unparseable)
train batches: 130  val: 17  test: 17


In [12]:
from tdc.single_pred import ADME
data = ADME(name="Half_Life_Obach")
df_diag = data.get_data()
print(df_diag["Y"].describe())
print(df_diag["Y"].quantile([0.90, 0.95, 0.99]))

Found local copy...
Loading...
Done!


count     667.000000
mean       18.212916
std        81.869185
min         0.065000
25%         1.750000
50%         4.200000
75%        11.000000
max      1200.000000
Name: Y, dtype: float64
0.90     30.4
0.95     45.7
0.99    200.0
Name: Y, dtype: float64


## 7. Model — `ADMET_MTL_Network`

Hard parameter sharing: one shared MLP trunk, five task-specific heads. Heads emit raw outputs (logits for classification, scalars for regression); `predict()` applies sigmoid for classification heads.

In [13]:
def _mlp_block(in_dim: int, out_dim: int, dropout: float) -> nn.Sequential:
    return nn.Sequential(
        nn.Linear(in_dim, out_dim),
        nn.LayerNorm(out_dim),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout),
    )


class _TaskHead(nn.Module):
    """Per-task MLP head producing a single scalar."""

    def __init__(self, in_dim: int, hidden_dim: int, dropout: float) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, shared_repr: torch.Tensor) -> torch.Tensor:
        return self.net(shared_repr).squeeze(-1)


class ADMET_MTL_Network(nn.Module):
    def __init__(
        self,
        tasks: Sequence[TaskSpec],
        input_dim: int = FP_NBITS,
        shared_hidden_dims: Sequence[int] = SHARED_HIDDEN_DIMS,
        head_hidden_dim: int = HEAD_HIDDEN_DIM,
        dropout: float = DROPOUT,
    ) -> None:
        super().__init__()
        self.tasks: Tuple[TaskSpec, ...] = tuple(tasks)
        self.task_names: Tuple[str, ...] = tuple(t.name for t in self.tasks)

        layers: List[nn.Module] = []
        prev = input_dim
        for h in shared_hidden_dims:
            layers.append(_mlp_block(prev, h, dropout))
            prev = h
        self.shared = nn.Sequential(*layers)
        shared_out_dim = prev

        # ModuleDict so heads are addressable by task name in the loss/log code.
        self.heads = nn.ModuleDict(
            {t.name: _TaskHead(shared_out_dim, head_hidden_dim, dropout) for t in self.tasks}
        )

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        shared = self.shared(x)
        return {name: head(shared) for name, head in self.heads.items()}

    @torch.no_grad()
    def predict(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        """Inference helper: applies sigmoid to classification heads."""
        self.eval()
        raw = self.forward(x)
        out: Dict[str, torch.Tensor] = {}
        for spec in self.tasks:
            y = raw[spec.name]
            out[spec.name] = torch.sigmoid(y) if spec.task_type == "classification" else y
        return out

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED) # dodane

model = ADMET_MTL_Network(TASKS)
print(model)

ADMET_MTL_Network(
  (shared): Sequential(
    (0): Sequential(
      (0): Linear(in_features=2048, out_features=1024, bias=True)
      (1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (2): ReLU(inplace=True)
      (3): Dropout(p=0.1, inplace=False)
    )
    (1): Sequential(
      (0): Linear(in_features=1024, out_features=512, bias=True)
      (1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (2): ReLU(inplace=True)
      (3): Dropout(p=0.1, inplace=False)
    )
    (2): Sequential(
      (0): Linear(in_features=512, out_features=256, bias=True)
      (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (2): ReLU(inplace=True)
      (3): Dropout(p=0.1, inplace=False)
    )
  )
  (heads): ModuleDict(
    (caco2): _TaskHead(
      (net): Sequential(
        (0): Linear(in_features=256, out_features=64, bias=True)
        (1): ReLU(inplace=True)
        (2): Dropout(p=0.1, inplace=False)
        (3): Linear(in_features=64, out_features=1, bias

## 8. Masked multi-task loss

**The critical bit.** Each molecule has labels for *some* tasks but not others. The mask zeroes out per-sample contributions for tasks that lack a ground-truth label, so no spurious gradient flows through those heads.

Per-task loss is averaged over the **number of valid labels for that task** (not batch size) — otherwise sparsely-labeled tasks get artificially shrunken losses and the optimizer ignores them. Total loss is the sum across tasks; per-task losses are returned separately for logging.

In [14]:
class MaskedMultiTaskLoss(nn.Module):
    def __init__(self, tasks: Sequence[TaskSpec]) -> None:
        super().__init__()
        self.tasks: Tuple[TaskSpec, ...] = tuple(tasks)
        self._task_index: Dict[str, int] = {t.name: i for i, t in enumerate(self.tasks)}

        # pos_weight: n_negative / n_positive dla niezbalansowanych klasyfikacji
        # < 1.0 oznacza że pozytywnych jest więcej niż negatywnych
        self._pos_weights: Dict[str, float] = {
                "bbb":    0.31,   # 465/1500
                "cyp3a4": 1.41,   # 7218/5110
                "herg":   0.45,   # 201/444
            }

    def forward(
        self,
        predictions: Dict[str, torch.Tensor],   # name -> (B,) raw outputs
        targets: torch.Tensor,                  # (B, n_tasks), NaNs already replaced with 0
        mask: torch.Tensor,                     # (B, n_tasks), 1 where labeled
    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        total = torch.zeros((), device=targets.device)
        per_task_log: Dict[str, torch.Tensor] = {}

        for spec in self.tasks:
            k = self._task_index[spec.name]
            y_pred = predictions[spec.name]      # (B,)
            y_true = targets[:, k]               # (B,)
            m = mask[:, k]                       # (B,) 1 if labeled, else 0

            # Per-sample loss WITHOUT reduction so we can apply the mask
            # element-wise. BCEWithLogitsLoss is numerically stabler than
            # sigmoid + BCE; for regression we use MSE.
            if spec.task_type == "classification":
                # pos_weight jako tensor na właściwym urządzeniu
                pw = self._pos_weights.get(spec.name, 1.0)
                pw_tensor = torch.tensor(pw, device=y_pred.device)
                per_sample = F.binary_cross_entropy_with_logits(
                    y_pred, y_true,
                    pos_weight=pw_tensor,
                    reduction="none"
                )
            else:
                #per_sample = F.mse_loss(y_pred, y_true, reduction="none")
                per_sample = F.huber_loss(y_pred, y_true, reduction="none", delta=1.0)

            # Zero out unlabeled samples' contributions for THIS task. Other
            # tasks see them normally if they have labels there.
            masked = per_sample * m

            # Normalize by number of valid samples (not batch size). Clamp
            # avoids div-by-zero when a batch has no labels for this task —
            # numerator is 0 anyway in that case, so loss is 0.
            n_valid = m.sum().clamp(min=1.0)
            task_loss = masked.sum() / n_valid
            weight = TASK_WEIGHTS.get(spec.name, 1.0)
            total = total + weight * task_loss
            #total = total + task_loss
            per_task_log[spec.name] = task_loss.detach()

        return total, per_task_log

In [15]:
print(df_merged["cyp3a4"].value_counts())
print(df_merged["herg"].value_counts())
print(df_merged["bbb"].value_counts())

cyp3a4
0.0    7218
1.0    5110
Name: count, dtype: int64
herg
1.0    444
0.0    201
Name: count, dtype: int64
bbb
1.0    1500
0.0     465
Name: count, dtype: int64


## 9. Trainer

Standard PyTorch loop with `AdamW`. Resolves the device through `config`, prints which one was picked, applies cuDNN/TF32 flags. Wraps the forward pass in `torch.cuda.amp.autocast` with a `GradScaler` when AMP is enabled (no-op on CPU). Logs total + per-task losses each epoch.

In [16]:
class _RunningMean:
    def __init__(self) -> None:
        self.sum: float = 0.0
        self.n: int = 0

    def update(self, value: float, count: int = 1) -> None:
        self.sum += value * count
        self.n += count

    @property
    def value(self) -> float:
        return self.sum / self.n if self.n else 0.0


class Trainer:
    def __init__(
        self,
        model: ADMET_MTL_Network,
        tasks: Sequence[TaskSpec],
        device: Optional[Union[torch.device, str]] = None,
        lr: float = LEARNING_RATE,
        weight_decay: float = WEIGHT_DECAY,
        use_amp: bool = USE_AMP,
    ) -> None:
        self.device = torch.device(device) if device is not None else resolve_device()
        configure_cuda_runtime()
        print(f"[trainer] using device: {self.device}")

        self.model = model.to(self.device)
        self.tasks = tuple(tasks)
        self.criterion = MaskedMultiTaskLoss(self.tasks).to(self.device)
        self.optimizer = AdamW(self.model.parameters(), lr=lr, weight_decay=weight_decay)

        self.use_amp: bool = use_amp and self.device.type == "cuda"
        self.scaler: torch.cuda.amp.GradScaler = torch.cuda.amp.GradScaler(enabled=self.use_amp)

    def _run_epoch(self, loader: DataLoader, train: bool) -> Dict[str, float]:
        self.model.train(train)
        total_meter = _RunningMean()
        per_task_meters: Dict[str, _RunningMean] = {t.name: _RunningMean() for t in self.tasks}

        ctx = torch.enable_grad() if train else torch.no_grad()
        with ctx:
            for fps, targets, mask in loader:
                fps = fps.to(self.device, non_blocking=True)
                targets = targets.to(self.device, non_blocking=True)
                mask = mask.to(self.device, non_blocking=True)

                with torch.cuda.amp.autocast(enabled=self.use_amp):
                    preds = self.model(fps)
                    total_loss, per_task = self.criterion(preds, targets, mask)

                if train:
                    self.optimizer.zero_grad(set_to_none=True)
                    self.scaler.scale(total_loss).backward()
                    self.scaler.step(self.optimizer)
                    self.scaler.update()

                bs = fps.size(0)
                total_meter.update(total_loss.item(), bs)
                for name, value in per_task.items():
                    per_task_meters[name].update(value.item(), bs)

        log: Dict[str, float] = {"total": total_meter.value}
        log.update({name: m.value for name, m in per_task_meters.items()})
        return log

    def fit(
        self,
        train_loader: DataLoader,
        val_loader: DataLoader,
        epochs: int = EPOCHS,
    ) -> None:
        for epoch in tqdm(range(1, epochs + 1), desc="epochs"):
            train_log = self._run_epoch(train_loader, train=True)
            val_log = self._run_epoch(val_loader, train=False)
            self._print_epoch(epoch, train_log, val_log)

    def _print_epoch(self, epoch: int, train_log: Dict[str, float], val_log: Dict[str, float]) -> None:
        head = f"[epoch {epoch:>3d}] train_total={train_log['total']:.4f} val_total={val_log['total']:.4f}"
        per_task_str = "  ".join(
            f"{t.name}: tr={train_log[t.name]:.4f}/va={val_log[t.name]:.4f}"
            for t in self.tasks
        )
        print(f"{head}  ::  {per_task_str}")

## 10. Train

In [17]:
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

trainer = Trainer(model, TASKS, device=DEVICE)
trainer.fit(train_loader, val_loader, epochs=EPOCHS)

[trainer] using device: cuda:0


epochs:   0%|          | 0/100 [00:00<?, ?it/s]

[epoch   1] train_total=7.3356 val_total=4.9477  ::  caco2: tr=1.3764/va=0.3074  bbb: tr=0.3214/va=0.2642  cyp3a4: tr=0.8076/va=0.7839  half_life: tr=0.5415/va=0.4641  herg: tr=0.4160/va=0.3474  hum_mic_cl: tr=0.3313/va=0.1657  ppb: tr=0.0489/va=0.0320  logd: tr=0.8356/va=0.6899  water_sol: tr=2.0440/va=1.0862
[epoch   2] train_total=4.1415 val_total=4.1705  ::  caco2: tr=0.3072/va=0.1795  bbb: tr=0.2656/va=0.2235  cyp3a4: tr=0.7151/va=0.6364  half_life: tr=0.3343/va=0.3924  herg: tr=0.2703/va=0.3290  hum_mic_cl: tr=0.2056/va=0.1649  ppb: tr=0.0254/va=0.0240  logd: tr=0.6537/va=0.5975  water_sol: tr=1.0109/va=0.8100
[epoch   3] train_total=2.8534 val_total=4.0800  ::  caco2: tr=0.2253/va=0.1660  bbb: tr=0.2008/va=0.1768  cyp3a4: tr=0.6019/va=0.5780  half_life: tr=0.2395/va=0.3772  herg: tr=0.1206/va=0.4066  hum_mic_cl: tr=0.1405/va=0.1306  ppb: tr=0.0237/va=0.0250  logd: tr=0.5155/va=0.4578  water_sol: tr=0.7692/va=0.6707
[epoch   4] train_total=2.1063 val_total=3.9907  ::  caco2: tr=0

## 11. Evaluate on held-out test set

ROC-AUC for classification heads, RMSE for regression heads. Metrics respect the label mask — molecules that lack a label for a given task are simply skipped for that task's metric.

In [18]:
@torch.no_grad()
def evaluate(
    model: ADMET_MTL_Network,
    loader: DataLoader,
    device: torch.device,
) -> None:
    model.eval()
    bucket: Dict[str, List[Tuple[float, float]]] = {t.name: [] for t in model.tasks}
    for fps, targets, mask in loader:
        fps = fps.to(device)
        preds = model.predict(fps)
        for k, spec in enumerate(model.tasks):
            m = mask[:, k].bool()
            if not m.any():
                continue
            y_pred = preds[spec.name].detach().cpu()[m].numpy()
            y_true = targets[:, k][m].numpy()
            bucket[spec.name].extend(zip(y_pred.tolist(), y_true.tolist()))

    print("\n[test] per-task metrics:")
    for spec in model.tasks:
        if not bucket[spec.name]:
            print(f"  - {spec.name}: no labels in this split")
            continue
        y_pred, y_true = map(np.array, zip(*bucket[spec.name]))
        if spec.task_type == "classification":
            try:
                metric = roc_auc_score(y_true, y_pred)
                print(f"  - {spec.name:<10s} ROC-AUC = {metric:.4f}  (n={len(y_true)})")
            except ValueError:
                print(f"  - {spec.name:<10s} ROC-AUC = n/a (single class)  (n={len(y_true)})")
        elif spec.task_type == "regression":
            # odwróć log1p dla half_life
            if spec.name == "half_life":
                y_pred = np.expm1(y_pred)
                y_true = np.expm1(y_true)
            rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
            print(f"  - {spec.name:<10s} RMSE = {rmse:.4f}  (n={len(y_true)})")  # dodane 
        else:
            rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
            print(f"  - {spec.name:<10s} RMSE    = {rmse:.4f}  (n={len(y_true)})")


evaluate(model, test_loader, trainer.device)


[test] per-task metrics:
  - caco2      RMSE = 0.4798  (n=95)
  - bbb        ROC-AUC = 0.8915  (n=228)
  - cyp3a4     ROC-AUC = 0.8764  (n=1269)
  - half_life  RMSE = 13.4027  (n=59)
  - herg       ROC-AUC = 0.8019  (n=49)
  - hum_mic_cl RMSE = 0.4351  (n=193)
  - ppb        RMSE = 0.2069  (n=130)
  - logd       RMSE = 0.7992  (n=1308)
  - water_sol  RMSE = 1.0633  (n=1167)


## 12. Quick sanity check: predict on a custom SMILES

Edit the list below and re-run. Classification heads return probabilities; regression heads return raw scalars.

In [19]:
custom_smiles = [
    "CC(=O)OC1=CC=CC=C1C(=O)O",        # aspirin
    "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",    # caffeine
    "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O",   # ibuprofen
]

fps, valid_idx = batch_smiles_to_fps(custom_smiles, radius=FP_RADIUS, n_bits=FP_NBITS)
fps_t = torch.from_numpy(fps).to(trainer.device)
preds = model.predict(fps_t)

rows = []
for i, orig_idx in enumerate(valid_idx):
    row = {"smiles": custom_smiles[orig_idx]}
    for spec in TASKS:
        v = preds[spec.name][i].item()
        row[f"{spec.name} ({spec.task_type[:3]})"] = round(v, 4)
    rows.append(row)

pd.DataFrame(rows)

,smiles,caco2 (reg),bbb (cla),cyp3a4 (cla),half_life (reg),herg (cla),hum_mic_cl (reg),ppb (reg),logd (reg),water_sol (reg)
0,CC(=O)OC1=CC=CC=C1C(=O)O,-5.1031,0.0082,0.0000,0.3210,0.0000,1.6761,0.2195,-3.0347,7.7316
1,CN1C=NC2=C1C(=O)N(C(=O)N2C)C,-4.3097,0.9990,0.0001,2.0398,0.9992,1.2682,0.3151,0.5489,7.3679
2,CC(C)CC1=CC=C(C=C1)C(C)C(=O)O,-4.9373,0.0017,0.0012,1.0972,0.0030,1.8837,0.6981,1.3388,5.2358
